<a href="https://colab.research.google.com/github/Amitabh-Phule/Deep-Learning/blob/main/DL_Exp10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Load and Preprocess MNIST Dataset

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
import numpy as np

print("Step 1: Loading MNIST...")

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Reduce dataset (IMPORTANT for speed + stability)
x_train = x_train[:10000]
y_train = y_train[:10000]

x_test = x_test[:2000]
y_test = y_test[:2000]

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Convert to 3 channels (MobileNetV2 expects 3 input channels)
x_train = np.stack((x_train,)*3, axis=-1)
x_test = np.stack((x_test,)*3, axis=-1)

# Resize (MobileNetV2 expects input size 96x96 for MobileNetV2-96)
x_train = tf.image.resize(x_train, (96, 96))
x_test = tf.image.resize(x_test, (96, 96))

Step 1: Loading MNIST...


### Step 2: Load Pretrained MobileNetV2 Model

In [3]:
print("Step 2: Loading Pretrained Model...")

# Load MobileNetV2 without the top (classification) layers
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96,96,3))

# Freeze base layers to prevent their weights from being updated during the first training phase
for layer in base_model.layers:
    layer.trainable = False

Step 2: Loading Pretrained Model...


### Step 3: Build and Compile the New Model

In [4]:
print("Step 3: Building Model...")

# Add custom classification head on top of the base model
x = base_model.output
x = layers.GlobalAveragePooling2D()(x) # Flatten the features
x = layers.Dense(128, activation='relu')(x) # Hidden dense layer
output = layers.Dense(10, activation='softmax')(x) # Output layer for 10 MNIST classes

model = models.Model(inputs=base_model.input, outputs=output)

# Compile the model for the initial training phase (with frozen base layers)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Step 3: Building Model...


### Step 4: Train the Model (Frozen Layers)

In [5]:
print("Step 4: Training (Frozen)...")

# Train only the newly added layers
model.fit(x_train, y_train, epochs=2, batch_size=32, validation_split=0.1)

Step 4: Training (Frozen)...
Epoch 1/2
282/282 ━━━━━━━━━━━━━━━━━━━━ 77s 248ms/step - accuracy: 0.9012 - loss: 0.2996 - val_accuracy: 0.9460 - val_loss: 0.1383
Epoch 2/2
282/282 ━━━━━━━━━━━━━━━━━━━━ 59s 211ms/step - accuracy: 0.9620 - loss: 0.1175 - val_accuracy: 0.9500 - val_loss: 0.1414


### Step 5: Evaluate Initial Performance

In [6]:
print("Step 5: Evaluating...")

# Evaluate the model's performance on the test set after initial training
loss, acc = model.evaluate(x_test, y_test)
print("Frozen Accuracy:", acc)

Step 5: Evaluating...
63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 177ms/step - accuracy: 0.9410 - loss: 0.1835
Frozen Accuracy: 0.9409999847412109


### Step 6: Fine-tuning (Unfreeze Layers)

In [7]:
print("Step 6: Fine-tuning...")

# Unfreeze the last 20 layers of the base model for fine-tuning
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Recompile the model with a very low learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5), # Lower learning rate is crucial for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training with fine-tuning
model.fit(x_train, y_train, epochs=1, batch_size=32)

Step 6: Fine-tuning...
313/313 ━━━━━━━━━━━━━━━━━━━━ 100s 286ms/step - accuracy: 0.8738 - loss: 0.4332


### Step 7: Evaluate Fine-tuned Performance

In [8]:
loss, acc = model.evaluate(x_test, y_test)
print("Fine-tuned Accuracy:", acc)

print("Experiment Completed Successfully")

63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 188ms/step - accuracy: 0.9405 - loss: 0.2088
Fine-tuned Accuracy: 0.940500020980835
Experiment Completed Successfully
